# Resume match ranking (whole-document version)

Ranks jobs in `job_tracker.db` by fit against `resume.md`, using a single whole-resume embedding compared against each whole job description -- no chunking, no truncation.

**How this differs from `resume_match_ranking.ipynb`**: that notebook splits the resume into sections/items and takes the *max* similarity across them, which answers "does *any one part* of my resume strongly match this job" -- useful for surfacing standout matches, but it means a single project in a weaker area (e.g. one data-engineering project) can make that whole job category look like a top fit, even if it's a small, non-representative slice of the overall resume. This notebook instead embeds the resume as one document and asks "does this job align with my resume's *overall* content" -- a job only scores well here if that domain is actually a meaningful share of the resume, not just present somewhere in it.

**Model**: `nomic-ai/nomic-embed-text-v1.5` instead of MiniLM -- supports up to 8192 tokens (vs. MiniLM's ~256), long enough to fit a full resume and most job descriptions without truncation, which is what makes whole-document embedding viable at all here. Requires `trust_remote_code=True` (runs model code pulled from the HF Hub alongside the weights -- a widely-used, actively-maintained model, but worth knowing what that flag means when you see the warning). Larger than MiniLM (~137M params vs ~22M) but still encoder-only, nowhere near the cost of something like an NLI cross-encoder.

Nomic's models were trained contrastively with an asymmetric query/document convention -- text needs a `"search_query: "` or `"search_document: "` prefix depending on its role. The resume is treated as the query ("find jobs matching this"), job postings as the documents being searched.

**Read-only**: reads from `job_tracker.db`, writes only a ranked CSV -- never touches the `jobs` table.

In [ ]:
import re
import sqlite3

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer, util

In [ ]:
# Config

DB_PATH = "job_tracker.db"   # run from repo root, per CLAUDE.md convention
RESUME_PATH = "resume.md"    # update to wherever your resume.md actually lives
MODEL_NAME = "nomic-ai/nomic-embed-text-v1.5"

QUERY_PREFIX = "search_query: "      # prefix for the resume (what we're matching FROM)
DOCUMENT_PREFIX = "search_document: "  # prefix for each job posting (what we're matching AGAINST)

EXCLUDE_EXPIRED = True   # drop is_expired = 1
EXCLUDE_APPLIED = True   # drop applied = 1 -- ranking is for deciding what to apply to next
TOP_N = 25

OUTPUT_CSV = "resume_match_simple_ranking.csv"

## Load and clean the resume

No section-splitting here -- the whole resume becomes one embedding input. `clean_markdown` still runs first so the model sees prose, not markup noise (`#`, `*`, `` ` `` etc.).

In [ ]:
def clean_markdown(text: str) -> str:
    """Light markdown-syntax strip so embeddings see prose, not markup noise."""
    text = re.sub(r"```.*?```", " ", text, flags=re.DOTALL)  # code blocks
    text = re.sub(r"!\[.*?\]\(.*?\)", " ", text)              # images
    text = re.sub(r"\[(.*?)\]\(.*?\)", r"\1", text)           # links -> link text
    text = re.sub(r"[#*`_>]", " ", text)                       # heading/emphasis/quote markers
    text = re.sub(r"-{3,}", " ", text)                         # horizontal rules
    text = re.sub(r"\s+", " ", text).strip()
    return text


with open(RESUME_PATH, "r", encoding="utf-8") as f:
    resume_raw = f.read()

resume_text = clean_markdown(resume_raw)
print(f"Resume length: {len(resume_text)} chars (~{len(resume_text.split())} words)")

## Load candidate jobs from the tracker

In [ ]:
conn = sqlite3.connect(DB_PATH)

where_clauses = []
if EXCLUDE_EXPIRED:
    where_clauses.append("(is_expired IS NULL OR is_expired = 0)")
if EXCLUDE_APPLIED:
    where_clauses.append("(applied IS NULL OR applied = 0)")
where_sql = f"WHERE {' AND '.join(where_clauses)}" if where_clauses else ""

query = f"""
    SELECT job_id, title, company, source, location, salary_str, work_arrangement,
           seniority, visa_eligibility, min_years_exp, job_url,
           is_agent, is_ai_llm, is_de, is_ds, is_swe,
           description
    FROM jobs
    {where_sql}
"""
jobs_df = pd.read_sql_query(query, conn)
conn.close()

jobs_df = jobs_df.dropna(subset=["description"]).reset_index(drop=True)
print(f"Candidate jobs: {len(jobs_df)}")

## Embed and score

One embedding for the whole resume, one embedding per job (title + description). `fit_score` is a single cosine similarity per job -- no max-pooling, since there's only one resume chunk now.

In [ ]:
model = SentenceTransformer(MODEL_NAME, trust_remote_code=True)

param_count = sum(p.numel() for p in model[0].auto_model.parameters())
print(f"Model params: {param_count:,} (~{param_count * 4 / (1024 ** 2):.0f} MB fp32)")
print(f"Max sequence length: {model.max_seq_length} tokens")

In [ ]:
resume_embedding = model.encode(QUERY_PREFIX + resume_text, convert_to_tensor=True)

job_texts = (jobs_df["title"].fillna("") + ". " + jobs_df["description"].fillna("")).apply(clean_markdown)
job_texts_prefixed = (DOCUMENT_PREFIX + job_texts).tolist()
job_embeddings = model.encode(job_texts_prefixed, convert_to_tensor=True, show_progress_bar=True)

similarities = util.cos_sim(resume_embedding, job_embeddings).cpu().numpy().flatten()
jobs_df["fit_score"] = similarities

## Ranked shortlist

In [ ]:
display_cols = [
    "job_id", "title", "company", "source", "fit_score",
    "salary_str", "work_arrangement", "seniority", "visa_eligibility",
    "is_agent", "is_ai_llm", "is_de", "is_ds", "is_swe", "job_url",
]

ranked = jobs_df.sort_values("fit_score", ascending=False).reset_index(drop=True)
ranked[display_cols].head(TOP_N)

In [ ]:
ranked[display_cols].to_csv(OUTPUT_CSV, index=False)
print(f"Wrote {len(ranked)} ranked rows to {OUTPUT_CSV}")

## Notes / caveats

- `fit_score` is uncalibrated for the same reason as the chunked version -- no labeled "good fit" ground truth exists, so only the *relative order* within a run is meaningful.
- This measures overall-resume alignment, which fixes the "one strong project dominates the ranking" issue from the chunked/max-pooled notebook -- but it trades away that notebook's ability to surface a standout niche match your resume's overall theme wouldn't suggest (e.g. one unusually strong, relevant side project in an otherwise unrelated resume). The two notebooks answer genuinely different questions; worth running both rather than treating one as strictly better.
- `trust_remote_code=True` runs code shipped alongside the model weights on the HF Hub, not just the weights themselves -- fine for a well-known, actively-maintained model like this one, but it's a different trust boundary than a plain-weights model like MiniLM, worth remembering if pointing this at a less-established model later.
- At ~8192 tokens, truncation is very unlikely for either the resume or most job descriptions here, but not literally impossible for an extreme outlier posting -- `model.max_seq_length` (printed above) confirms the actual limit in effect.
- Same read-only design as the chunked notebook: writes a CSV, never writes back into `job_tracker.db`.